# Automatic Test-Time Inference Demo

## 🎯 Goal: Predict GIST class from CT scan ONLY (no manual mask needed!)

**Pipeline:**
1. CT Scan (image.nii.gz)
2. → SAM-Med3D (auto-segment tumor)
3. → GeoTopo-STS (extract features)
4. → Prediction (tumor class)

---

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\cahel\Desktop\Med3Tab-PFN')

import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 1. Initialize Auto-Inference Pipeline

In [ ]:
from geotopo_sts.sam_geotopo_integration import GeoTopoSTS_AutoInference

# Paths
sam_checkpoint = r'C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\work_dir\SAM\sam_med3d.pth'
geotopo_checkpoint = r'C:\Users\cahel\Desktop\Med3Tab-PFN\geotopo_sts\outputs\gist_experiment\best_model.pth'
config_path = r'C:\Users\cahel\Desktop\Med3Tab-PFN\geotopo_sts\config.yaml'

# Initialize pipeline
pipeline = GeoTopoSTS_AutoInference(
    geotopo_config_path=config_path,
    geotopo_checkpoint_path=geotopo_checkpoint,
    sam_checkpoint_path=sam_checkpoint,
    sam_model_type='vit_b',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print("\n✅ Pipeline ready for automatic inference!")

## 2. Test on a New GIST Case (No Manual Mask!)

In [ ]:
# Select a test case
test_case = r'C:\Users\cahel\Desktop\Med3Tab-PFN\data\gist\GIST-010_CT\1\NIFTI\image.nii.gz'

# Option 1: Fully automatic (no human input)
print("\n" + "="*60)
print("FULLY AUTOMATIC INFERENCE")
print("="*60)

prediction_auto = pipeline.predict_from_file(test_case)

print(f"\n✅ Final prediction: Class {prediction_auto}")

## 3. Test with Prompt Point (More Accurate)

In [ ]:
# If you know roughly where the tumor is, provide a point
# Format: (z_slice, y_coord, x_coord)

print("\n" + "="*60)
print("INFERENCE WITH PROMPT POINT")
print("="*60)

# Example: tumor center at slice 50, position (150, 150)
tumor_location = (50, 150, 150)  # Adjust based on your data

prediction_prompted = pipeline.predict_from_file(
    test_case,
    prompt_point=tumor_location
)

print(f"\n✅ Final prediction: Class {prediction_prompted}")

## 4. Visualize Auto-Generated Mask

In [ ]:
# Load CT scan
nii = nib.load(test_case)
volume = nii.get_fdata().astype(np.float32)
spacing = tuple(nii.header.get_zooms()[:3])

# Predict and get mask
prediction, auto_mask = pipeline.predict(
    volume, spacing, 
    prompt_point=tumor_location,
    return_mask=True
)

print(f"\nPrediction: Class {prediction}")
print(f"Auto-mask shape: {auto_mask.shape}")
print(f"Tumor volume: {auto_mask.sum() * np.prod(spacing) / 1000:.2f} cm³")

In [ ]:
# Visualize middle slice
z_mid = volume.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CT slice
axes[0].imshow(volume[z_mid], cmap='gray', vmin=-150, vmax=350)
axes[0].set_title('CT Scan (Input)')
axes[0].axis('off')

# Auto-generated mask
axes[1].imshow(auto_mask[z_mid], cmap='hot')
axes[1].set_title('Auto-Segmented Mask\n(SAM-Med3D)')
axes[1].axis('off')

# Overlay
axes[2].imshow(volume[z_mid], cmap='gray', vmin=-150, vmax=350)
axes[2].imshow(auto_mask[z_mid], cmap='hot', alpha=0.3)
axes[2].set_title(f'Overlay\nPrediction: Class {prediction}')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 5. Compare with Ground Truth (if available)

In [ ]:
# Load manual segmentation for comparison
gt_mask_path = test_case.replace('image.nii.gz', 'segmentation.nii.gz')

if Path(gt_mask_path).exists():
    gt_mask = nib.load(gt_mask_path).get_fdata().astype(np.uint8)
    
    # Compute Dice score
    intersection = (auto_mask * gt_mask).sum()
    dice = 2 * intersection / (auto_mask.sum() + gt_mask.sum())
    
    print(f"\n📊 Segmentation Quality:")
    print(f"   Dice Score: {dice:.3f}")
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(gt_mask[z_mid], cmap='Greens')
    axes[0].set_title('Ground Truth (Expert)')
    axes[0].axis('off')
    
    axes[1].imshow(auto_mask[z_mid], cmap='Reds')
    axes[1].set_title('Auto-Segmented (SAM)')
    axes[1].axis('off')
    
    # Overlap
    overlap = np.zeros((*gt_mask[z_mid].shape, 3))
    overlap[gt_mask[z_mid] > 0] = [0, 1, 0]  # Green
    overlap[auto_mask[z_mid] > 0] = [1, 0, 0]  # Red
    overlap[(gt_mask[z_mid] > 0) & (auto_mask[z_mid] > 0)] = [1, 1, 0]  # Yellow (overlap)
    
    axes[2].imshow(overlap)
    axes[2].set_title(f'Overlap (Dice: {dice:.3f})\nGreen=GT, Red=Auto, Yellow=Both')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No ground truth available for comparison")

## 6. Batch Test on Multiple Cases

In [ ]:
from geotopo_sts.gist_data_loader import discover_gist_cases
from tqdm import tqdm

# Find all GIST cases
gist_root = r'C:\Users\cahel\Desktop\Med3Tab-PFN\data\gist'
cases = discover_gist_cases(gist_root)

# Test on first 10 cases
test_cases = cases[:10]

print(f"\n" + "="*60)
print(f"BATCH TEST: {len(test_cases)} CASES")
print("="*60)

results = []
for case_info in tqdm(test_cases):
    try:
        prediction = pipeline.predict_from_file(case_info['image_path'])
        results.append({
            'case_id': case_info['case_id'],
            'prediction': prediction
        })
    except Exception as e:
        print(f"Error on {case_info['case_id']}: {e}")

# Show results
print("\n📊 Results:")
for res in results:
    print(f"  {res['case_id']:15s} → Class {res['prediction']}")

## Summary

✅ **You can now predict GIST class from CT scans ONLY!**

### Training:
- Uses expert manual masks (`segmentation.nii.gz`)
- Learns optimal features from geometry + topology

### Test Time:
- **No manual masks needed!**
- SAM-Med3D automatically segments tumors
- GeoTopo-STS extracts features and predicts

### Performance Tips:
1. **Fully automatic**: Fast but may miss small tumors
2. **With prompt**: Single click → better segmentation
3. **Fine-tune SAM**: Train SAM-Med3D on your GIST data for best results

---

**Your method is now clinically viable!** 🚀